#### Import Libraries 

In [31]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import random
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from typing import Tuple, Dict, List, Optional
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    f1_score,
    average_precision_score,
)
import optuna
import optuna.visualization as vis

optuna.logging.set_verbosity(optuna.logging.WARNING)
import sklearn, catboost, matplotlib

print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn", sklearn.__version__)
print("optuna", optuna.__version__)
print("catboost", catboost.__version__)
print("shap", shap.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)

Python: 3.13.12 (main, Feb  4 2026, 09:25:39) [GCC 11.4.0]
numpy: 2.4.2
pandas: 3.0.1
sklearn 1.8.0
optuna 4.7.0
catboost 1.2.10
shap 0.50.0
matplotlib: 3.10.8
seaborn: 0.13.2


#### Reproducible configuration

In [3]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

#### Clean CSV file

In [4]:
input_file = "dataset/bank-full.csv"
output_file = "dataset/cleaned_bank_full.csv"

with open(input_file, 'r', newline='', encoding='utf-8') as dirty, \
     open(output_file, 'w', newline='', encoding='utf-8') as clean:
    for line in dirty:
        cleaned_line = line.replace('"', '')
        cleaned_line = line.replace(';', ',')
        clean.write(cleaned_line)   

#### Load dataset

In [5]:
df = pd.read_csv("dataset/cleaned_bank_full.csv")
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no


#### Convert strings to integers

In [6]:
for c in df.columns:
    if df[c].dtype == "str":
        lbl = LabelEncoder()
        lbl.fit(list(df[c].values))
        df[c] = lbl.transform(list(df[c].values))

df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,4,1,2,0,2143,1,0,2,5,8,261,1,-1,0,3,0
1,44,9,2,1,0,29,1,0,2,5,8,151,1,-1,0,3,0
2,33,2,1,1,0,2,1,1,2,5,8,76,1,-1,0,3,0
3,47,1,1,3,0,1506,1,0,2,5,8,92,1,-1,0,3,0
4,33,11,2,3,0,1,0,0,2,5,8,198,1,-1,0,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,9,1,2,0,825,0,0,0,17,9,977,3,-1,0,3,1
45207,71,5,0,0,0,1729,0,0,0,17,9,456,2,-1,0,3,1
45208,72,5,1,1,0,5715,0,0,0,17,9,1127,5,184,3,2,1
45209,57,1,1,1,0,668,0,0,1,17,9,508,4,-1,0,3,0


#### Separate features / target

In [7]:
X = df.drop('y', axis=1)
y = df['y']

#### Target information

In [8]:
target_details = y.value_counts()
print(f"Unsuccessful Bank Marketing: {target_details[0]}")
print(f"Successful Banking Marketing: {target_details[1]}")
print(f"Ratio: {target_details[0] / target_details[1]:.1f}")

Unsuccessful Bank Marketing: 39922
Successful Banking Marketing: 5289
Ratio: 7.5


#### Split train/test

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

#### Objective function for Optuna

In [10]:
# Calculamos scale_pos_weight exacto basado en desbalance
pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print("Scale pos weight final:", pos_weight)


def objective(trial):
    param = {
        "iterations": trial.suggest_int("iterations", 1900, 2100),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.025, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 9, 10),
        "border_count": trial.suggest_int("border_count", 90, 130),
        "depth": 6,
        "scale_pos_weight": pos_weight,
        "random_state": RANDOM_STATE,
        "eval_metric": "AUC",
        "loss_function": "Logloss",
        "verbose": 0,
    }

    skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
    auc_scores = []
    fold_idx = []

    for i, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model = CatBoostClassifier(**param)
        model.fit(
            X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=0
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, y_pred_proba))
        trial.report(roc_auc_score(y_val, y_pred_proba), i)
        fold_idx.append(i)
        trial.set_user_attr("fold", fold_idx)
        trial.set_user_attr("scores", auc_scores)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(auc_scores)

Scale pos weight final: 7.548333727251241


#### Optuna: Hyperparameter Search

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(),
)
study.optimize(
    objective, n_trials=5, show_progress_bar=True
) 

print("Mejores hiperparámetros encontrados:")
for k, v in study.best_trial.params.items():
        print(f"  {k}: {v}")


Best trial: 4. Best value: 0.937061: 100%|██████████| 5/5 [04:25<00:00, 53.06s/it]

Mejores hiperparámetros encontrados:
  iterations: 1946
  learning_rate: 0.01821624156352625
  l2_leaf_reg: 9.730366895207009
  border_count: 109


In [12]:
# optuna.visualization.plot_optimization_history(study)

#### Optimization history

![Optimization History](outputs/optimization_history.png)

In [13]:
# vis.plot_intermediate_values(study).show()

#### Intermediate values

![Intermediate values](outputs/intermediate_values.png)

#### Final training

In [14]:
best_model = CatBoostClassifier(
    **study.best_params,
    depth= 6,
    eval_metric="AUC",
    loss_function="Logloss",
    random_state=RANDOM_STATE,
    scale_pos_weight=pos_weight,
    verbose=0
)

best_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

CatBoostClassifier(border_count=109, depth=6, eval_metric='AUC', iterations=1946, l2_leaf_reg=9.730366895207009, learning_rate=0.01821624156352625, loss_function='Logloss', random_state=42, scale_pos_weight=7.548333727251241, verbose=0)

#### Visualization of Optuna convergence in 3 graphs

In [ ]:
def plot_optuna_convergence(study: optuna.Study):
    """
    1. AUC por trial con mejor acumulado → muestra velocidad de convergencia
    2. Importancia de hiperparámetros (fANOVA) → qué parámetros importan más
    3. Histograma de distribución de AUCs → calidad general de la búsqueda
    """
    completed = [
        t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
    ]  # Trials terminados
    pruned = [
        t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED
    ]  # Trials podados

    trial_nums = [t.number for t in completed]  # Índices de trials completados
    trial_aucs = [t.value for t in completed]  # AUC de cada trial completado

    # AUC máximo acumulado hasta cada trial (curva de convergencia)
    best_so_far = [max(trial_aucs[: i + 1]) for i in range(len(trial_aucs))]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Convergencia de Optuna - CatBoost", fontsize=13, fontweight="bold")

    # AUC por trial + mejor acumulado
    ax1 = axes[0]
    ax1.scatter(
        trial_nums,
        trial_aucs,
        alpha=0.5,
        s=30,
        color="steelblue",
        label="AUC por trial",
        zorder=2,
    )
    ax1.plot(
        trial_nums,
        best_so_far,
        color="darkorange",
        lw=2.5,
        label="Mejor acumulado",
        zorder=3,
    )

    best_t = study.best_trial  # Trial con mejor AUC global
    ax1.scatter(
        best_t.number,
        best_t.value,
        color="red",
        s=200,
        zorder=4,
        marker="*",
        label=f"Mejor #{best_t.number} ({best_t.value:.4f})",
    )
    ax1.axhline(y=0.93, color="gray", ls="--", lw=1.5, label="Target AUC=0.93")

    ax1.set_xlabel("Trial #")
    ax1.set_ylabel("ROC AUC (CV 3-fold)")
    ax1.set_title(f"Convergencia\n{len(completed)} completados | {len(pruned)} podados")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    # Importancia de hiperparámetros (fANOVA) 
    ax2 = axes[1]
    try:
        # fANOVA calcula qué fracción de la varianza del AUC explica cada param
        importances = optuna.importance.get_param_importances(study)
        params_list = list(importances.keys())[:10]  # Top 10 parámetros
        vals_list = [importances[p] for p in params_list]

        # Ordenar de mayor a menor
        sorted_pairs = sorted(zip(vals_list, params_list), reverse=True)
        vals_list, params_list = zip(*sorted_pairs)

        bars = ax2.barh(
            range(len(params_list)),
            vals_list,
            color="steelblue",
            alpha=0.8,
            edgecolor="black",
            lw=0.5,
        )

        for bar, val in zip(bars, vals_list):
            ax2.text(
                bar.get_width() + 0.003,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}",
                va="center",
                fontsize=8,
            )

        ax2.set_yticks(range(len(params_list)))
        ax2.set_yticklabels(params_list, fontsize=9)
        ax2.set_xlabel("Importancia relativa (fANOVA)")
        ax2.set_title("Importancia de Hiperparámetros")
        ax2.grid(True, alpha=0.3, axis="x")

    except Exception:
        ax2.text(
            0.5,
            0.5,
            "Importancia no disponible\n(requiere más trials)",
            ha="center",
            va="center",
            transform=ax2.transAxes,
        )

    # Histograma de AUCs 
    ax3 = axes[2]
    if len(trial_aucs) > 1:
        ax3.hist(
            trial_aucs,
            bins=min(20, len(trial_aucs) // 2 + 1),
            color="steelblue",
            alpha=0.7,
            edgecolor="black",
            lw=0.5,
        )
        ax3.axvline(
            x=study.best_value,
            color="red",
            lw=2,
            ls="--",
            label=f"Mejor: {study.best_value:.4f}",
        )
        ax3.axvline(
            x=np.mean(trial_aucs),
            color="orange",
            lw=1.5,
            ls=":",
            label=f"Media: {np.mean(trial_aucs):.4f}",
        )
        ax3.axvline(x=0.93, color="gray", lw=1.5, ls="-.", label="Target: 0.93")

    ax3.set_xlabel("ROC AUC")
    ax3.set_ylabel("Número de trials")
    ax3.set_title("Distribución de AUCs\n(Todos los trials completados)")
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    #plt.savefig("outputs/optuna_convergence.png", dpi=150, bbox_inches="tight")
    #plt.show()


plot_optuna_convergence(study)

#### Convergencia de Optuna - CatBoost

![Convergencia de Optuna - CatBoost](outputs/optuna_convergence.png)

#### Predictions and probabilities

In [16]:
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

#### Adjusting threshold using F1

In [17]:
prec, rec, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]
print("Threshold ajustado para balancear:", best_threshold)

y_pred_adjusted = (y_pred_proba >= best_threshold).astype(int)

Threshold ajustado para balancear: 0.712494073768237


#### Metrics and confusion matrix

In [32]:
print("Classification Report:")
print(classification_report(y_test, y_pred_adjusted))

cm = confusion_matrix(y_test, y_pred_adjusted)
print(cm)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix con threshold ajustado + scale_pos_weight")
plt.xlabel("Predicted")
plt.ylabel("Actual")
#plt.savefig('outputs/confusion_matrix_ajustado.png', dpi=150, bbox_inches='tight')
#plt.show()

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.92      0.94      7985
           1       0.56      0.75      0.64      1058

    accuracy                           0.90      9043
   macro avg       0.76      0.84      0.79      9043
weighted avg       0.92      0.90      0.91      9043

[[7352  633]
 [ 265  793]]


Text(244.44444444444446, 0.5, 'Actual')

#### Confusion matrix

![Confusion matrix](outputs/confusion_matrix_ajustado.png)

In [19]:
roc_auc = roc_auc_score(y_test, y_pred_proba)
print("ROC AUC:", roc_auc)

ROC AUC: 0.9335799756869272


#### Curva ROC (Receiver Operating Characteristic)

In [33]:
def plot_roc_curve(
    y_true: np.ndarray,
    y_probs: np.ndarray,
    title: str = "ROC Curve"
) -> float:
    """
    Muestra el trade-off entre TPR y FPR para todos los thresholds.
    AUC = 1.0 → clasificador perfecto | AUC = 0.5 → aleatorio.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)  # Calcular puntos de la curva
    auc_score = roc_auc_score(y_true, y_probs)           # Área bajo la curva

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(fpr, tpr, color='darkorange', lw=2.5,
            label=f'ROC AUC = {auc_score:.4f}')
    ax.plot([0, 1], [0, 1], 'navy', lw=1.5, ls='--',
            label='Clasificador aleatorio (AUC=0.5)')
    ax.fill_between(fpr, tpr, alpha=0.1, color='darkorange')   # Área sombreada

    # Punto óptimo: maximiza índice de Youden (TPR - FPR)
    opt_idx = np.argmax(tpr - fpr)
    ax.scatter(fpr[opt_idx], tpr[opt_idx], color='red', s=120, zorder=5,
               label=f'Punto óptimo (FPR={fpr[opt_idx]:.3f}, TPR={tpr[opt_idx]:.3f})')

    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=12)
    ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=12)
    ax.set_title(f'{title}\nROC AUC = {auc_score:.4f}', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    #plt.savefig('outputs/roc_curve.png', dpi=150, bbox_inches='tight')
    #plt.show()
    print(f"  📊 ROC AUC: {auc_score:.4f}")
    return auc_score
final_auc = plot_roc_curve(y_test, y_pred_proba, "ROC Curve - Test Set")

  📊 ROC AUC: 0.9336


#### Roc curve

![Roc curve](outputs/roc_curve.png)

#### Roc auc per fold

In [34]:
fold_idx = np.round(study.trials[0].user_attrs["fold"], 4)
auc_scores = np.round(study.trials[0].user_attrs["scores"], 4)
fig, ax = plt.subplots()
bars = ax.bar(fold_idx, auc_scores)

ax.bar_label(bars, labels=[f"{v}" for v in auc_scores], padding=1)
plt.xlabel("Fold")
plt.ylabel("ROC AUC")
plt.title("ROC AUC por fold - Stratified 4-Fold CV")
#plt.savefig("outputs/roc_auc_fold_stratified.png", dpi=150, bbox_inches="tight")
#plt.show()

print("AUC mean:", np.mean(auc_scores))
print("AUC std:", np.std(auc_scores))

AUC mean: 0.937
AUC std: 0.002534758371127323


#### Roc Auc 4 Fold Stratified

![Roc Auc 4 Fold Stratified](outputs/roc_auc_fold_stratified.png)

#### Precision-Recall Curve + F1 vs Threshold.

In [35]:
def plot_precision_recall(
    y_true: np.ndarray,
    y_probs: np.ndarray,
    threshold: float
):
    """
    Más informativa que ROC para datasets desbalanceados porque
    no incluye los Verdaderos Negativos (muy numerosos en desbalance).
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)
    ap_score = average_precision_score(y_true, y_probs) # AP = área bajo curva PR
    baseline = y_true.mean()                             # Prevalencia clase positiva

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Precision-Recall - Test Set', fontsize=13, fontweight='bold')

    # Curva PR 
    ax1 = axes[0]
    ax1.step(recall, precision, where='post', color='steelblue', lw=2.5,
             label=f'Average Precision = {ap_score:.4f}')
    ax1.fill_between(recall, precision, step='post', alpha=0.15, color='steelblue')
    ax1.axhline(y=baseline, color='red', ls='--',
                label=f'Baseline (prevalencia={baseline:.3f})')

    if len(thresholds) > 0:
        idx = np.argmin(np.abs(thresholds - threshold))    # Índice más cercano
        ax1.scatter(recall[idx], precision[idx], color='red', s=150,
                    zorder=5, label=f'Threshold={threshold:.3f}')

    ax1.set_xlabel('Recall (Sensibilidad)', fontsize=12)
    ax1.set_ylabel('Precision (VPP)', fontsize=12)
    ax1.set_title(f'Curva Precision-Recall\nAP = {ap_score:.4f}', fontsize=12)
    ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)
    ax1.set_xlim([0, 1]); ax1.set_ylim([0, 1.05])

    # F1 vs Threshold
    ax2 = axes[1]
    thresh_range = np.linspace(0.01, 0.99, 200)    # 200 umbrales candidatos
    f1_scores = [
        f1_score(y_true, (y_probs >= t).astype(int), zero_division=0)
        for t in thresh_range
    ]

    ax2.plot(thresh_range, f1_scores, color='green', lw=2.5, label='F1-Score')
    ax2.axvline(x=threshold, color='red', ls='--',
                label=f'Threshold óptimo = {threshold:.3f}')
    best_idx = np.argmax(f1_scores)
    ax2.scatter(thresh_range[best_idx], f1_scores[best_idx],
                color='red', s=150, zorder=5,
                label=f'F1 máximo = {f1_scores[best_idx]:.4f}')

    ax2.set_xlabel('Threshold de Clasificación', fontsize=12)
    ax2.set_ylabel('F1-Score', fontsize=12)
    ax2.set_title('F1-Score vs Threshold', fontsize=12)
    ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    #plt.savefig('outputs/precision_recall_curve.png', dpi=150, bbox_inches='tight')
    #plt.show()
    print(f"  📊 Average Precision: {ap_score:.4f}")

plot_precision_recall(y_test, y_pred_proba, best_threshold)

  📊 Average Precision: 0.6341


#### Precision Recall - Test Set

![Precision recall curve](outputs/precision_recall_curve.png)

#### Visualizing the importance of CatBoost's native features

In [36]:
def plot_catboost_feature_importance(
    model: CatBoostClassifier,
    feature_names: List[str]
):
    """
    CatBoost proporciona 3 tipos de importancia:
        PredictionValuesChange: Cambio en predicción al perturbar la feature
        LossFunctionChange:     Cambio en la función de pérdida
        FeatureImportance:      Importancia por ganancia (similar a XGBoost)

    Aquí se usa PredictionValuesChange por ser la más interpretable.
    """
    # Obtener importancias del modelo
    importances  = model.get_feature_importance()   # PredictionValuesChange
    feature_imp  = pd.DataFrame({
        'feature':    feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)   # Ordenar de mayor a menor

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = ['#e74c3c' if i < 5 else '#3498db'     # Top-5 en rojo, resto en azul
              for i in range(len(feature_names))]

    bars = ax.barh(
        range(len(feature_imp)),
        feature_imp['importance'].values,
        color=colors[::-1],         # Invertir colores para match con el orden
        alpha=0.8, edgecolor='black', lw=0.5
    )

    # Anotar valores de importancia
    for i, (bar, val) in enumerate(zip(bars, feature_imp['importance'].values[::-1])):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=9)

    ax.set_yticks(range(len(feature_imp)))
    ax.set_yticklabels(feature_imp['feature'].values[::-1], fontsize=10)
    ax.set_xlabel('Importancia (PredictionValuesChange)', fontsize=11)
    ax.set_title('Importancia de Features - CatBoost\n'
                 '(Top-5 en rojo)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    #plt.savefig('outputs/catboost_feature_importance.png', dpi=150, bbox_inches='tight')
    #plt.show()

    # Mostrar top-5 en consola
    print(f"\n  📋 Top-5 Features por Importancia Nativa CatBoost:")
    for i, row in feature_imp.head(5).iterrows():
        print(f"     {feature_imp.index.get_loc(i)+1}. "
              f"{row['feature']}: {row['importance']:.4f}")

feature_names = X.columns

plot_catboost_feature_importance(best_model, feature_names)


  📋 Top-5 Features por Importancia Nativa CatBoost:
     1. duration: 27.6376
     2. month: 21.9445
     3. contact: 10.1321
     4. day: 8.1074
     5. poutcome: 6.4194


#### Features importance

![Features importance](outputs/catboost_feature_importance.png)

#### SHAP

In [24]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)
rng = np.random.default_rng(seed=42)

In [25]:
# Top 5 features más influyentes
importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values(by='mean_abs_shap', ascending=False)
print("Top 5 features más importantes:")
print(importance_df.head(5))

Top 5 features más importantes:
     feature  mean_abs_shap
11  duration       1.612497
8    contact       0.583685
10     month       0.534473
6    housing       0.348521
5    balance       0.206394


#### SHAP Bar Plot 
**Importancia Global Promedio |SHAP| sobre el dataset**

In [37]:
fig = shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=10, rng=rng, show=False)
#plt.savefig("outputs/shap_summary_bar.png", dpi=150, bbox_inches='tight')
#plt.show()

#### Shap summary bar

![Shap summary bar](outputs/shap_summary_bar.png)

In [38]:
fig = shap.summary_plot(shap_values, X_test, plot_type="dot", max_display=10, rng=rng, show=False)
#plt.savefig("outputs/shap_summary_beeswarm.png", dpi=150, bbox_inches='tight')
#plt.show()

#### SHAP Beeswarm
**Cada punto = una muestra. Color = valor de la feature. Posición X = impacto en predicción**

![Shap beeswarm](outputs/shap_summary_beeswarm.png)

In [28]:
fig = shap.waterfall_plot(shap.Explanation(values=shap_values[0],
                                     base_values=explainer.expected_value,
                                     data=X_test.iloc[0]), show=False)
#plt.savefig("outputs/shap_summary_waterfall.png", dpi=150, bbox_inches='tight')
#plt.show()

#### SHAP Waterfall - Muestra #0
**Contribución acumulada de cada feature desde el valor base**

![Shap waterfall](outputs/shap_summary_waterfall.png)

In [29]:
fig = shap.force_plot(explainer.expected_value, shap_values[0], X_test.iloc[0], matplotlib=True, show=False)
#plt.savefig("outputs/shap_summary_force.png", dpi=150, bbox_inches='tight')
#plt.show()

#### SHAP Force - Muestra #0
**Rojo = aumenta predicción | Azul = reduce predicción**

![Shap force](outputs/shap_summary_force.png)

#### Predictions

In [30]:
predictions_df = pd.DataFrame()
predictions_df['Actual'] = y_test
predictions_df['Predicted'] = y_pred
predictions_df['probability'] = y_pred_proba
predictions_df['Match'] = predictions_df['Actual'] == predictions_df['Predicted']
predictions_df['Check'] = predictions_df['Match'].map({True: '✅', False: '❌'})   
predictions_df.sample(10)

,Actual,Predicted,probability,Match,Check
38382,0,0,0.054665,True,✅
36363,0,0,0.006448,True,✅
25534,0,0,0.142806,True,✅
35299,0,0,0.131131,True,✅
20082,1,1,0.893565,True,✅
27965,0,0,0.021947,True,✅
34665,0,0,0.234108,True,✅
20195,0,0,0.032674,True,✅
15606,0,0,0.200586,True,✅
8208,0,0,0.024699,True,✅
